# Exemplo de uso — `redshift_etl` no JupyterHub

Fluxo completo: **discover** (descobre e persiste o modelo do schema) → **extract** (lê paginado e grava parquet particionado) → **transform** (carrega qualquer tabela como `DataFrame` pandas).

Este notebook assume que o pacote `redshift_etl` está disponível (repositório clonado no ambiente do JupyterHub).

In [ ]:
import sys

# Se o notebook estiver em notebooks/ na raiz do repo, adiciona a raiz ao path
sys.path.append("..")

import pandas as pd

from redshift_etl import discover, load_model, extract_all, TableLoader

## Conexão com o Redshift

O pacote não abre conexão sozinho: ele espera receber uma função `query(sql) -> pandas.DataFrame`. Ajuste a célula abaixo para a forma de conexão já usada no seu JupyterHub (aqui um exemplo com `redshift_connector`, lendo credenciais de variáveis de ambiente — nunca deixe usuário/senha hardcoded no notebook).

In [ ]:
import os
import redshift_connector

conn = redshift_connector.connect(
    host=os.environ["REDSHIFT_HOST"],
    database=os.environ["REDSHIFT_DATABASE"],
    user=os.environ["REDSHIFT_USER"],
    password=os.environ["REDSHIFT_PASSWORD"],
    port=int(os.environ.get("REDSHIFT_PORT", 5439)),
)


def query(sql: str) -> pd.DataFrame:
    with conn.cursor() as cursor:
        cursor.execute(sql)
        return cursor.fetch_dataframe()

## Parâmetros do job

`PAGE_SIZE` e `NUM_BUCKETS` controlam o trade-off tempo x memória x custo: páginas maiores reduzem o número de round-trips ao Redshift (menos overhead, menos tempo de leader node), enquanto mais buckets paralelizam melhor a leitura posterior em pandas/Spark. Ajuste conforme o volume de cada schema.

In [ ]:
SCHEMA = "meu_schema"
CONFIG_DIR = "./config"          # onde ficam o modelo e as queries persistidas (JSON)
OUTPUT_DIR = "./dados_extraidos"  # onde ficam os parquets
PAGE_SIZE = 100_000
NUM_BUCKETS = 32

## 1. Discover — descobrir o modelo do schema

Busca tabelas, colunas e contagem de linhas via `information_schema`, e já persiste o resultado em `CONFIG_DIR/<schema>.json`. Rode isso uma vez; nas próximas sessões você pode pular direto para a célula "fluxo alternativo" abaixo, sem gastar leader node do Redshift de novo.

In [ ]:
model = discover(query, SCHEMA, config_dir=CONFIG_DIR)

pd.DataFrame(
    [{"table_name": t, "columns": len(cols), "row_count": model.row_counts[t]} for t, cols in model.tables.items()]
).sort_values("row_count", ascending=False)

### Fluxo alternativo — retomar de uma configuração já salva

Em uma sessão nova (ex.: kernel reiniciado, outro dia), se `CONFIG_DIR/<schema>.json` já existe, não é preciso repetir o `discover` contra o Redshift: basta carregar o modelo salvo. Descomente a linha abaixo para usar esse caminho em vez da célula anterior.

In [ ]:
# model = load_model(SCHEMA, config_dir=CONFIG_DIR)
# model.tables_with_rows()

## 2. Extract — ler paginado e gravar parquet particionado

Só tabelas com ao menos 1 linha são lidas. Cada tabela é ordenada e paginada pela melhor chave disponível (`date_modified`/`date_created` + `id`/`id_c`, com fallback para `id`/`id_c`), e a query usada fica salva em `CONFIG_DIR/<schema>/<tabela>.json` para auditoria/reprodução.

In [ ]:
relatorio = extract_all(
    query, model, OUTPUT_DIR, page_size=PAGE_SIZE, num_buckets=NUM_BUCKETS, config_dir=CONFIG_DIR
)
relatorio

## 3. Transform — trabalhar as tabelas extraídas em pandas

`TableLoader` encapsula a leitura dos parquets particionados; a partir daí é pandas puro.

In [ ]:
loader = TableLoader(OUTPUT_DIR, schema=SCHEMA)
loader.available_tables()

In [ ]:
df_accounts = loader["accounts"]  # atalho equivalente a loader.load("accounts")
df_accounts.head()

In [ ]:
# Carregando só as colunas necessárias (menos I/O, mais rápido em tabelas largas)
df_leads = loader.load("custom_leads_c", columns=["id_c", "score"])
df_leads.describe()

## Notas de custo/performance

- **Reexecução barata**: se `CONFIG_DIR` já tem o modelo e as queries, use `load_model` para pular o `discover` — evita bater no Redshift só para remontar o que já é conhecido.
- **`PAGE_SIZE`**: valores maiores reduzem o número de queries (menos overhead de leader node), mas aumentam o pico de memória local por página lida.
- **`NUM_BUCKETS`**: mais buckets favorecem paralelismo na leitura posterior (Spark, Athena, etc.); para leitura só em pandas, 32 (default) costuma ser suficiente.
- **`loader.load(table, columns=[...])`**: carregue apenas as colunas necessárias em tabelas largas — o parquet é colunar, então isso reduz I/O de verdade, não só memória.